In this notebook we'll explore different document loaders (for parsing) and splitters (for chunking) available through `langchain`.

# Parsing

## PyPDF loader

This is suited to simple, clean PDFs.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# doc1 contains 4 columns per page, so I'm testing whether the loader can handle this layout

doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/motion_sickness_PIL.pdf"

# doc 2 is a more standard layout, with 1 column per page

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/antacid_PIL.pdf"

In [ ]:
loader1 = PyPDFLoader(doc1_path)
document1 = loader1.load()

In [ ]:
loader2 = PyPDFLoader(doc2_path)
document2 = loader2.load()

### Explore the extracted page content

In [ ]:
for page_num, page in enumerate(document1):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

This document loader correctly read down each column before moving to the next one. However, one interesting artifact is that at the bottom of the first column it extracts the numbers from the section headings which go across the page horizontally. This might make it difficult for the RAG system to refer to a specific section number.

In [ ]:
for page_num, page in enumerate(document2):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

The second document looks good. The only mistake I noticed is that the loader added a space in the middle of the word 'quantitative'. I don't think there's a fix for this, but I don't think it's too much of a problem since I imagine an LLM will understand that the space isn't meant to be there.

### Cleaning page content

In document 2, there's a lot of whitespace e.g between section headings and the text in the given section. We could remove this whitespace so that the section heading and text are more likely to appear in the same chunk, which helps preserve context and also reduces token usage.

Let's write a function to clean the page content:
- Collapse whitespace, i.e multiple spaces or tabs back-to-back
- Wherever 3 or more new lines occur consecutively, replace them with 2 new lines

In particular, the gaps between paragraphs look like `\n \n \n` (i.e there is a space between new lines), so we'll need to account for that in the regex.

The new line processing is informed by the fact that LangChain's `RecursiveCharacterTextSplitter` -- which we'll use for chunking -- looks for `\n\n` as the plain-text representation of a paragraph break.

In [ ]:
import re


def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)  # collapse repeated spaces/tabs
    text = re.sub(r"[ \t]+\n", "\n", text)  # strip trailing space before a newline
    text = re.sub(
        r"\n{3,}", "\n\n", text
    )  # collapse blank-line runs to one paragraph break
    return text.strip()


assert clean_text(" Hello   World ") == "Hello World"
assert clean_text("Hello\n\n\nWorld") == "Hello\n\nWorld"
assert clean_text("Hello\n \n \nWorld") == "Hello\n\nWorld"

There are no line breaks in document 1, so let's test this cleaning function on document 2.

In [ ]:
loader = PyPDFLoader(doc2_path)
documents = loader.load()

doc2_page_content = documents[1].page_content
print(doc2_page_content)

In [ ]:
cleaned_content = clean_text(doc2_page_content)
print(cleaned_content)

As expected the multiple-line-breaks are reduced to single line breaks.

### Explore the extracted metadata

In [ ]:
for page_num, page in enumerate(documents):
    print(f"=== Page {page_num + 1} ===")
    print(page.metadata)
    print()

Looking at the metadata, there are a few things that would be useful to persist to the chunking step:
- `source`: currently this gives the whole file path, but we're just interested in the file name
- `page`: this will enable citations referencing the specific page number on which the chunk appears in the original document

We're not interested in metadata such as `producer`, `creator` and `moddate` as they don't relate to the document's content.

## Directory loader

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
directory_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/"

loader = DirectoryLoader(
    directory_path, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
)
documents = loader.load()

print(f"Number of Documents: {len(documents)}")


for idx, document in enumerate(documents, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

This loads all documents in a given directory. This could be useful for my pipeline, but to start with I think I'll specify each document path separately.

## Classify page elements

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

This loader is suited to PDFs with complex layouts, or scanned documents which require OCR to extract the text. It classifies page elements such as titles, tables, paragraphs etc.

In [ ]:
loader1 = UnstructuredPDFLoader(doc1_path)
documents1 = loader1.load()

print(f"Number of Documents: {len(documents1)}")

for idx, document in enumerate(documents1, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For document 1 this loader seems to extract an inconsistent line length. It also puts multiple bullet points onto the same line, which seems strange.

In [ ]:
loader2 = UnstructuredPDFLoader(doc2_path)
documents2 = loader2.load()

print(f"Number of Documents: {len(documents2)}")

for idx, document in enumerate(documents2, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For document 2, each section number is put on a separate line from the section heading, which didn't happen with PyPDFLoader.

Even if this loader did work better than simpler ones, we'd need to consider the extra processing time it takes.

## Future Work

#### Scaling to more documents

I am starting with a small dataset that I can manually inspect, but this wouldn't scale if I was working with 1000s of documents. In that case, the quality of the parse becomes more important in order to reduce the need for format-specific cleaning.


#### Golden test set

With a larger dataset I'd label a small set of samples with diverse layouts. Each time I make a change to the parsing/cleaning pipeline I could look at how this affects the samples in the golden data set.


#### Detecting anomalies

I could select features such as chunk length, lines with single characters etc and calculate the distribution of these features across all documents. I could then manually investigate anomalies.

The important thing to come back to is the effect of parsing & cleaning on retrieval. I'd have a golden dataset of question-answer pairs with associated relevant chunks in order to evaluate retrieval, and I could measure any changes that result from changes to parsing / cleaning.


# Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from rag_pipeline.parse_documents import parse_document

My initial concern from looking at how formats differ between leaflets in my dataset is that the amount of text in each section can vary a lot. For example, the sections in document 1 contain a lot of text, while in document 2 they're much shorter, often a single line of text per section. 

If I split the documents into fixed-size chunks, then some chunks will contain a lot more sections than others. Since sections in theory represent distinct ideas / pieces of information, this means chunks could end up containing too many distinct ideas. If one of many small sections within a chunk contained information relevant to a query, then the embedding for the whole chunk might not represent that information very well and it might not be retrieved.

Let's have a look at the two example documents I chose to demonstrate this issue:

In [ ]:
clean_text_1 = parse_document(doc1_path)

doc_1_page_1_clean_text = clean_text_1[0].page_content
print(doc_1_page_1_clean_text)

In [ ]:
clean_text_2 = parse_document(doc2_path)

doc_2_page_1_clean_text = clean_text_2[0].page_content
print(doc_2_page_1_clean_text)

Let's apply a fixed-size splitter, e.g `RecursiveCharacterTextSplitter`, and have a look at the resulting chunks.

Here's how I'd go about choosing `chunk_size` & `chunk_overlap` for this splitter:

- In order to set `chunk_size`, we need to take into account the maximum sequence length of the embedding model that we'll be using. We can revisit this once we've chosen an embedding model, but for now let's work off of the assumption that 256 is a safe number of tokens for most sentence-transformer models. Assuming ~4 characters per token, this gives a chunk size of ~1000.
    - Note that in our production-ready code, parameters such as `chunk_size` will be set in a config file, not hardcoded.
- It is generally recommended that the overlap between chunks should be set to 10-20% of the chunk size. We'll take 15% of 1000 which is 150. It's good to remember that the purpose of this overlap is to maintain context between one chunk and the next. Without overlap, an idea could get split between two chunks. However, setting the overlap too high wastes storage space in the vector store and wastes compute during retrieval.
- `RecursiveCharacterTextSplitter` also has a `separators` parameter which defaults to `["\n\n", "\n", " ", ""]`. This means it first tries to split on paragraphs, then new lines, then words, then characters.

In [ ]:
doc_1_page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(doc_1_page_1_clean_text)

In [ ]:
doc_2_page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(doc_2_page_1_clean_text)

In [ ]:
for chunk in doc_1_page_1_chunks:
    print(chunk)
    print("====================")

In [ ]:
for chunk in doc_2_page_1_chunks:
    print(chunk)
    print("====================")

For document 1 we can see that a single section is sometimes split over multiple chunks, whereas for document 2 chunks contain multiple sections.

If we wanted 1 section per chunk, how could we do that?

In document 1 the main section headings are numbered 1-6, but within these sections there are sometimes sub-sections which are not numbered but which appeared in bold text in the original document.

In document 2 each section is numbered 1-10 and sub-sections are numbered too (e.g 4.1, 4.2).

We can't just define a regex to detect sections in both documents. I think we need to revisit the parsing step and use a loader that detects section headers with various formats, whether numbered or bold.

# ?

LangChain splitters also have a `split_documents` attribute. The resulting chunks are `langchain_core.documents.base.Document` objects, just like the outputs of our `parse_document` function. This way we can persist the metadata from pages to chunks.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

for document in clean_text:
    for chunk_index, chunk in enumerate(splitter.split_documents([document])):
        print(chunk_index, chunk.metadata)